# Fitting Normal-1D Mixture Distribution to Data with Uncertainties

## Imports

In [ ]:
import numpy as np  # import numpy for constructing mock data and initial conditions
import essrs        # import rust-based sampler machinery
import matplotlib.pyplot as plt
import math
import scipy.stats as scistat
import time

## Define parameters

In [ ]:
N_COMPONENTS: int = 2
N_PARAMETERS_PER_COMPONENT: int = 2

In [ ]:
N_WALKERS: int = 100        # the number of walkers running in the ensemble

N_PARAMETERS: int = (
    (N_COMPONENTS - 1) + N_COMPONENTS * N_PARAMETERS_PER_COMPONENT
)       # the number of parameters in the fit

# MCMC specific constants
N_BURN_IN: int = 200
N_SAMPLES: int = 1000

In [ ]:
N_DATA_POINTS = np.random.randint(2000, 4000, N_COMPONENTS)
POPULATION_SIGMA = np.random.uniform(1, 3, N_COMPONENTS)

POPULATION_MEAN = [0.0]
for (a, b) in zip(POPULATION_SIGMA[:-1], POPULATION_SIGMA[1:]):
    POPULATION_MEAN.append((a+b) * np.sqrt(2))
POPULATION_MEAN = np.array(POPULATION_MEAN)

norm = np.sum(N_DATA_POINTS)

# this defines the convolution distribution
# because we are dispersing the errors we cannot simply take naive mean and subtract error in quadrature
ERROR_CENTROID: float = 2.0 # the mean error which will be applied to points
ERROR_SIGMA: float = 0.1    # standard deviation around mean of error which is applied to points

print(N_DATA_POINTS)
print(POPULATION_MEAN)
print(POPULATION_SIGMA)

## Generate mock data

The print statements at the bottom highlight the point made earlier.
The mean is only minorly different after convoltion (and the difference may even be a sampling effect from sampling the points).
The standard deviation however is significantly larger highlighting the issue with taking naive estimates of the parameters.

In [ ]:
data = np.array([])
error = np.array([])

for n, mu, sigma in zip(N_DATA_POINTS, POPULATION_MEAN, POPULATION_SIGMA):
    data = np.concatenate((
        data,
        np.random.normal(mu, sigma, n),
    ))
    error = np.concatenate((
        error,
        np.random.normal(ERROR_CENTROID, ERROR_SIGMA, n)
    ))

idx_sort = np.argsort(data)
data = data[idx_sort]
error = error[idx_sort]

# apply a convolution by shifting each data point from true value by sampling from error
convolution = error * np.random.normal(0.0, 1.0, len(error))
conv_data = data + convolution

## Generate initial conditions

The naive estimates are good candidates for initial conditions of the fit.
The initial conditions are generated by taking the mean and sigma values and randomly distributing points around those values.

In [ ]:
ics = np.empty(shape=(N_WALKERS, 0))

for _ in range(N_COMPONENTS-1):
    ics = np.column_stack((
        ics,
        np.random.normal(1/N_COMPONENTS, 0.01,N_WALKERS)
    ))

n_per_normal = math.floor(len(data) / N_COMPONENTS)

for i in range(N_COMPONENTS):
    _slice = data[i*n_per_normal:(i+1) * n_per_normal]
    mu = np.mean(_slice)
    sig = np.std(_slice)

    ics = np.column_stack((
        ics,
        np.random.normal(mu, 0.01, N_WALKERS),
        np.random.normal(sig, 0.01, N_WALKERS)
    ))


## MCMC fit

In [ ]:
model = essrs.model.scientific.normal_1d_mix_deconv(conv_data, error, N_COMPONENTS)

burn_move = essrs.moves.hybrid_ddc_move(one_in_n=10)
sampler_move = essrs.moves.differential_move()
config = essrs.EnsembleConfig(n_samples=N_SAMPLES, n_walkers=N_WALKERS, n_parameters=N_PARAMETERS, n_burn_in=N_BURN_IN, n_threads=None)

start = time.perf_counter()
(fit, _) = essrs.run_mcmc(
    log_likelihood=model,
    config=config,
    burn_in_move=burn_move,
    sampling_move=sampler_move,
    initial_conditions=ics
)

elapsed = time.perf_counter() - start
print(f"ESSRS took {elapsed:.3f} seconds")

In [ ]:
x = np.linspace(np.amin(conv_data), np.amax(conv_data), 10000)
dx = x[1] - x[0]

plt.figure()
plt.hist(data, histtype="step", bins=100, density=True, color="b", zorder=10)
plt.hist(conv_data, histtype="step", bins=100, density=True, color="r", zorder=10)

for n, mu, sig in zip(N_DATA_POINTS, POPULATION_MEAN, POPULATION_SIGMA):
    y = n / norm * scistat.norm.pdf(x=x, loc=mu, scale=sig)
    plt.plot(x,y, color="k", ls=":")


for params in fit[-1, :, :]:
    thetas = params[:N_COMPONENTS-1]
    thetas = np.concatenate((thetas, np.array([1 - np.sum(thetas)])))


    y_all = np.zeros_like(x)
    for k, theta in enumerate(thetas):
        
        li = N_COMPONENTS-1 + 2*k 
        ri = N_COMPONENTS-1 + 2*k + 1
        y = theta * scistat.norm.pdf(x=x, loc=params[li], scale=params[ri])
        plt.plot(x,y, color="g", alpha=0.05)

        y_all += y
            
    plt.plot(x,y_all, color="k", alpha=0.2)


plt.show()